# CNN-TCN — Niño 3.4 (ONI) Forecasting  ·  Objective 1

Deep-learning forecaster that predicts the Niño 3.4 ONI at a fixed lead time from a
12-month window of four spatial anomaly fields over the tropical Pacific.

**Architecture (per the proposal):**
- **Stage 1 — Spatial Encoder (CNN):** a 2-D CNN applied *independently* to each of the
  12 monthly snapshots `(C=4, H=31, W=81)`, producing one spatial feature vector per month.
- **Stage 2 — Temporal Encoder (TCN):** stacked **dilated causal** 1-D convolutions
  (dilation `1, 2, 4, ...`) model the 12-month evolution; the last time step is the
  compact encoded representation `z`.
- **Stage 3 — ENSO Head (MLP):** regresses a single scalar — the ONI at `+LEAD` months.

This notebook reuses the same tensor-building pipeline, split, and train-only standardization
as `EDA_oras5.ipynb`, but feeds the model **six pre-standardised (`_z`) anomaly channels**
(SST, SLP, zonal/meridional wind stress, upper-300 m OHC, 20 °C-isotherm depth).
The South-Asia impact head (Objective 2) is intentionally left out here and can be added later.

> Run from the `data_analysis/` folder so the relative `data/...` CSV paths resolve.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 1 · Config

Inputs are the **six pre-standardised (`_z`) anomaly channels**: SST, SLP, zonal (`iews`)
and meridional (`inss`) wind stress, upper-300 m ocean heat content, and 20 °C-isotherm

depth. `LEAD=3` matches the XGBoost baseline; set it to `6` to align with the proposal's6-month objective (retrain to compare).

In [ ]:
# Six pre-standardised (_z) anomaly channels: SST, SLP, zonal & meridional wind
# stress, upper-300m ocean heat content, and 20C-isotherm depth.
FEATURES   = ['sst_anom_z', 'msl_anom_z', 'avg_iews_anom_z', 'avg_inss_anom_z',
              'sohtc300_anom_z', 'so20chgt_anom_z']
SEQ_LEN    = 12            # months per input window
LEAD       = 6             # forecast horizon (months after the window's last month)
TARGET_COL = 'nino34_3m'   # 3-month running-mean Niño 3.4 = official ONI

PACIFIC_CSV = 'data/Combined/combined_era5_oras5.csv'
NINO_CSV    = 'data/Combined/nino34_index_3month_running_mean_official_oni.csv'

BATCH_SIZE = 32
EPOCHS     = 300
LR         = 1e-3

WEIGHT_DECAY = 1e-4
CKPT_PATH  = 'cnn_tcn_enso_best.pt'
PATIENCE   = 40            # early-stopping patience (epochs w/o val improvement)

## 2 · Load data & build the spatio-temporal tensor `(T, H, W, C)`

Same reindex-onto-a-complete-grid logic as the EDA notebook: any missing cell becomes NaN
(guaranteeing correct ordering), ocean-only land cells are recorded, then NaNs are filled
with `0.0` (a neutral anomaly).

In [ ]:
pacific_df = pd.read_csv(PACIFIC_CSV)
nino = pd.read_csv(NINO_CSV)

times = np.sort(pacific_df['time'].unique())
lats  = np.sort(pacific_df['latitude'].unique())
lons  = np.sort(pacific_df['longitude'].unique())
T, H, W, C = len(times), len(lats), len(lons), len(FEATURES)
print(f'T={T} months | H={H} lats | W={W} lons | C={C} channels')

full_idx = pd.MultiIndex.from_product([times, lats, lons],
                                      names=['time', 'latitude', 'longitude'])
gridded = (pacific_df
           .set_index(['time', 'latitude', 'longitude'])[FEATURES]
           .reindex(full_idx))

grid = np.full((T, H, W, C), np.nan, dtype=np.float32)
for c, feat in enumerate(FEATURES):
    grid[..., c] = gridded[feat].to_numpy(dtype=np.float32).reshape(T, H, W)

nan_fraction = np.isnan(grid).mean(axis=(0, 1, 2))
grid = np.nan_to_num(grid, nan=0.0)
print('Full grid tensor (T, H, W, C):', grid.shape)
for feat, frac in zip(FEATURES, nan_fraction):
    print(f'  {feat:<16} NaN filled: {frac*100:5.1f}%')

## 3 · Align the ONI target and build 12-month sliding windows

Window `i` covers months `[i, i+SEQ_LEN-1]`; its target is the ONI `LEAD` months after the
window's last month. Warm-up NaNs (from the 3-month running mean) are skipped.

In [ ]:
target_series = nino.set_index('time')[TARGET_COL]
target_arr = target_series.reindex(times).to_numpy(dtype=np.float32)

X_list, y_list, sample_times = [], [], []
last_start = T - SEQ_LEN - LEAD + 1
for i in range(last_start):
    t_idx = i + SEQ_LEN - 1 + LEAD
    y_val = target_arr[t_idx]
    if np.isnan(y_val):
        continue
    X_list.append(grid[i:i + SEQ_LEN])     # (SEQ_LEN, H, W, C)
    y_list.append(y_val)
    sample_times.append(times[t_idx])

X = np.stack(X_list).astype(np.float32)    # (N, SEQ_LEN, H, W, C)
y = np.asarray(y_list, dtype=np.float32)   # (N,)
sample_dt = pd.to_datetime(np.asarray(sample_times))

assert np.isfinite(X).all() and np.isfinite(y).all()
print('X:', X.shape, '| y:', y.shape)
print('Predicted months:', sample_dt.min().date(), '->', sample_dt.max().date())

## 4 · Chronological split + per-channel re-standardization (train stats only)

No shuffling — this is a time series. The `_z` inputs are already scaled, but they were
z-scored over the *full* record; here we **re-standardize using train samples only**
(across sample, time, lat, lon) so the scaling itself is leakage-free, then apply to all splits.

In [ ]:
train_mask = sample_dt <= pd.Timestamp('2018-12-31')
val_mask   = (sample_dt >= pd.Timestamp('2019-01-01')) & (sample_dt <= pd.Timestamp('2022-12-31'))
test_mask  = sample_dt >= pd.Timestamp('2023-01-01')
for name, m in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
    dts = sample_dt[m]
    print(f'{name:5s}: n={m.sum():4d}  {dts.min().date()} -> {dts.max().date()}')

mu = X[train_mask].mean(axis=(0, 1, 2, 3))         # (C,)
sd = X[train_mask].std(axis=(0, 1, 2, 3)) + 1e-8   # (C,)
X_std = (X - mu) / sd
print('\nPer-channel mean:', np.round(mu, 4))
print('Per-channel std :', np.round(sd, 4))

## 5 · To PyTorch tensors + `DataLoader`s

Conv2d expects channels-first, so we permute `(N, T, H, W, C) -> (N, T, C, H, W)`.

In [ ]:
def to_tensor(mask):
    xt = torch.from_numpy(X_std[mask]).permute(0, 1, 4, 2, 3).contiguous()  # (N,T,C,H,W)
    yt = torch.from_numpy(y[mask]).float()
    return xt, yt

X_tr, y_tr = to_tensor(train_mask)
X_va, y_va = to_tensor(val_mask)
X_te, y_te = to_tensor(test_mask)
print('X_tr:', tuple(X_tr.shape), '| X_va:', tuple(X_va.shape), '| X_te:', tuple(X_te.shape))

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE)
test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=BATCH_SIZE)

## 6 · Stage 1 — Spatial Encoder (CNN)

A small 2-D CNN applied **identically to every month**. We fold the time axis into the batch
dimension (`B*T`), run the convolutions once, then unfold back to a per-month feature
sequence `F ∈ R^(B x T x d)`.

In [ ]:
class SpatialEncoder(nn.Module):
    """Applies a 2-D CNN independently to each monthly snapshot -> feature vector of dim `d`."""
    def __init__(self, in_ch=4, d=128):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                                   # 31x81 -> 15x40
            nn.Conv2d(32, 64, 3, padding=1),   nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                                   # 15x40 -> 7x20
            nn.Conv2d(64, d, 3, padding=1),    nn.BatchNorm2d(d),  nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),                           # -> (d, 1, 1)
        )
        self.d = d

    def forward(self, x):                 # x: (B, T, C, H, W)
        B, Tm, Cc, Hh, Ww = x.shape
        x = x.reshape(B * Tm, Cc, Hh, Ww)  # fold time into batch
        f = self.cnn(x).reshape(B, Tm, self.d)  # (B, T, d)
        return f

## 7 · Stage 2 — Temporal Encoder (TCN)

Stacked **dilated causal** 1-D convolutions. Left-padding + a "chomp" keep the convolution
causal (step `t` never sees `t+1`); dilation grows as `1, 2, 4, ...` so the receptive field
covers the whole 12-month window. The **last** time step becomes the encoded vector `z`.

In [ ]:
class Chomp1d(nn.Module):
    """Removes the extra right-hand padding to enforce causality."""
    def __init__(self, chomp): super().__init__(); self.chomp = chomp
    def forward(self, x): return x[:, :, :-self.chomp].contiguous() if self.chomp > 0 else x


class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.net(x)
        res = x if self.down is None else self.down(x)
        return self.relu(out + res)          # residual connection


class TemporalEncoder(nn.Module):
    def __init__(self, in_dim, channels=(128, 64, 64), kernel_size=3, dropout=0.1):
        super().__init__()
        layers, prev = [], in_dim
        for l, ch in enumerate(channels):
            layers.append(TemporalBlock(prev, ch, kernel_size, dilation=2 ** l, dropout=dropout))
            prev = ch
        self.tcn = nn.Sequential(*layers)
        self.out_dim = prev

    def forward(self, f):                    # f: (B, T, d)
        z = self.tcn(f.transpose(1, 2))      # -> (B, ch, T)
        return z[:, :, -1]                   # last time step -> (B, ch) = z

## 8 · Full model — CNN → TCN → ENSO head

The Stage-3 ENSO head is a small MLP regressing the ONI scalar. (A second impact head can
share the same `z` later for Objective 2.)

In [ ]:
class CNN_TCN(nn.Module):
    def __init__(self, in_ch=4, d=128, tcn_channels=(128, 64, 64), head_hidden=64, dropout=0.1):
        super().__init__()
        self.spatial  = SpatialEncoder(in_ch=in_ch, d=d)
        self.temporal = TemporalEncoder(in_dim=d, channels=tcn_channels, dropout=dropout)
        self.enso_head = nn.Sequential(
            nn.Linear(self.temporal.out_dim, head_hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(head_hidden, 1),
        )

    def forward(self, x):                    # x: (B, T, C, H, W)
        f = self.spatial(x)                  # (B, T, d)
        z = self.temporal(f)                 # (B, h)
        return self.enso_head(z).squeeze(-1) # (B,)


model = CNN_TCN(in_ch=C).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {n_params:,}')

## 9 · Training loop (MSE) with early stopping

Adam + `ReduceLROnPlateau`; the weights with the lowest validation RMSE are checkpointed.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=15)


def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * xb.size(0); n += xb.size(0)
    return (total / n) ** 0.5              # RMSE


best_val, best_epoch, wait = float('inf'), -1, 0
hist = {'train': [], 'val': []}
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    scheduler.step(va)
    hist['train'].append(tr); hist['val'].append(va)
    if va < best_val - 1e-5:
        best_val, best_epoch, wait = va, epoch, 0
        torch.save(model.state_dict(), CKPT_PATH)
    else:
        wait += 1
    if epoch % 10 == 0 or epoch == 1:
        print(f'epoch {epoch:3d}  train RMSE={tr:.4f}  val RMSE={va:.4f}  (best {best_val:.4f} @ {best_epoch})')
    if wait >= PATIENCE:
        print(f'Early stop at epoch {epoch} (no val improvement for {PATIENCE} epochs).')
        break

print(f'\nBest val RMSE={best_val:.4f} at epoch {best_epoch}. Restored from {CKPT_PATH}.')
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist['train'], label='train RMSE')
plt.plot(hist['val'],   label='val RMSE')
plt.axvline(best_epoch - 1, color='r', ls='--', lw=0.8, label=f'best @ {best_epoch}')
plt.xlabel('epoch'); plt.ylabel('RMSE (ONI)'); plt.title('CNN-TCN training curve')
plt.legend(); plt.tight_layout(); plt.show()

## 10 · Evaluation — RMSE · MAE · Pearson r · R²

Same metrics as the XGBoost baseline for a like-for-like comparison.

In [ ]:
from scipy.stats import pearsonr


@torch.no_grad()
def predict(xt):
    model.eval()
    out = []
    for i in range(0, xt.size(0), BATCH_SIZE):
        out.append(model(xt[i:i + BATCH_SIZE].to(device)).cpu())
    return torch.cat(out).numpy()


def report(name, y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae  = np.mean(np.abs(y_true - y_pred))
    r    = pearsonr(y_true, y_pred)[0]
    r2   = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
    print(f'{name:5s}  RMSE={rmse:.4f}  MAE={mae:.4f}  Pearson r={r:.4f}  R2={r2:.4f}')
    return dict(rmse=rmse, mae=mae, r=r, r2=r2)


pred_tr = predict(X_tr); pred_va = predict(X_va); pred_te = predict(X_te)
print(f'CNN-TCN | lead={LEAD} months\n')
m_tr = report('train', y_tr.numpy(), pred_tr)
m_va = report('val',   y_va.numpy(), pred_va)
m_te = report('test',  y_te.numpy(), pred_te)
cnntcn_metrics = {'train': m_tr, 'val': m_va, 'test': m_te, 'lead': LEAD}

## 11 · Actual vs predicted ONI over the full timeline

In [ ]:
pred_full = predict(torch.from_numpy(X_std).permute(0, 1, 4, 2, 3).contiguous())

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sample_dt, y, 'k-', lw=1.6, label='Actual ONI')
ax.plot(sample_dt, pred_full, color='C0', lw=1.2, label='CNN-TCN predicted')
ax.axhline(0.5,  color='r', ls=':', lw=0.8)
ax.axhline(-0.5, color='b', ls=':', lw=0.8)
ax.axvspan(pd.Timestamp('2019-01-01'), pd.Timestamp('2022-12-31'), color='orange', alpha=0.10, label='validation')
ax.axvspan(pd.Timestamp('2023-01-01'), sample_dt.max(),            color='green',  alpha=0.10, label='test')
ax.set_title(f'Nino 3.4 ONI - Actual vs CNN-TCN (lead={LEAD} mo)')
ax.set_xlabel('Predicted month'); ax.set_ylabel('ONI (degC)')
ax.legend(loc='upper left', ncol=2); plt.tight_layout(); plt.show()

## 12 · Save the trained model

Best weights are already at `CKPT_PATH`. On Colab, copy this file to Google Drive so it
survives runtime resets.

In [ ]:
torch.save({'state_dict': model.state_dict(),
            'mu': mu, 'sd': sd,
            'config': {'FEATURES': FEATURES, 'SEQ_LEN': SEQ_LEN, 'LEAD': LEAD,
                       'H': H, 'W': W, 'C': C}},
           'cnn_tcn_enso_full.pt')
print('Saved -> cnn_tcn_enso_full.pt  (weights + normalization stats + config)')